# Download Economist Page Images from METS

This notebook downloads page images for a supplied list of Economist archive page IDs. Page IDs use the `yyyy-mmdd-pppp` convention, for example `1867-0105-0027`.

Run this notebook from its own directory, `code/scripts`. The repository's VS Code setting `jupyter.notebookFileRoot = ${fileDirname}` makes that the default in VS Code/Jupyter.

Default input:

`../../data/processed/top_100_pre_1900_face_confidence_pages.json`

Default output directory:

`../../data/images/top_100_conf_pre_1900_default/`

Authentication cookies are read from `../auth/nationallizenzen_cookie.json`.

## Parameters

To download a custom set of pages, set `PAGE_IDS` to a list such as `["1867-0105-0027"]` or to a whitespace/comma-separated string. Leave it as `None` to load `INPUT_JSON`. `IMAGE_SIZE` selects the METS `fileGrp USE` value; available values depend on the issue metadata and commonly include `DEFAULT`, `MIN`, `MAX`, and `THUMB`.

In [ ]:
from pathlib import Path


# Set PAGE_IDS to a list/string for ad hoc runs, or leave None to load INPUT_JSON.
PAGE_IDS = None
INPUT_JSON = Path("../../data/processed/top_100_pre_1900_face_confidence_pages.json")
OUTPUT_DIR = Path("../../data/images/top_100_conf_pre_1900_default/")
IMAGE_SIZE = "DEFAULT"
OVERWRITE = False

METADATA_ROOT = Path("../../data/metadata")
AUTH_COOKIE_PATH = Path("../auth/nationallizenzen_cookie.json")
REQUEST_TIMEOUT = 60
REQUEST_SLEEP_SECONDS = 0.1


In [ ]:
import json
import re
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests


pd.options.display.max_rows = 120
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 180

page_id_re = re.compile(r"^(?P<issue_id>\d{4}-\d{4})-(?P<page_number>\d{4})$")
INPUT_JSON = Path(INPUT_JSON)
OUTPUT_DIR = Path(OUTPUT_DIR)
METADATA_ROOT = Path(METADATA_ROOT)
AUTH_COOKIE_PATH = Path(AUTH_COOKIE_PATH)
image_size = str(IMAGE_SIZE).upper()
download_manifest_json = OUTPUT_DIR / "download_manifest.json"

AUTH_FAILURE_MARKERS = (
    "Nationallizenzen Web Anmeldedienst",
    "login.nationallizenzen.de",
    "SAML2/Redirect/SSO",
)
XLINK_HREF = "{http://www.w3.org/1999/xlink}href"

assert image_size, "IMAGE_SIZE must not be empty."
assert REQUEST_TIMEOUT > 0
assert REQUEST_SLEEP_SECONDS >= 0


## Load and Validate Page List

The page list may come from the notebook parameter `PAGE_IDS` or from a JSON file. In both cases the normalized result must be a non-empty list of unique `yyyy-mmdd-pppp` identifiers.

In [ ]:
def parse_page_ids(value):
    if value is None:
        loaded = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
    elif isinstance(value, Path):
        loaded = json.loads(value.read_text(encoding="utf-8"))
    elif isinstance(value, str):
        possible_path = Path(value)
        if possible_path.exists() and possible_path.suffix.lower() == ".json":
            loaded = json.loads(possible_path.read_text(encoding="utf-8"))
        else:
            loaded = [part for part in re.split(r"[\s,;]+", value.strip()) if part]
    else:
        loaded = list(value)

    assert isinstance(loaded, list), "Page input must resolve to a list."
    page_ids = [str(page_id).strip() for page_id in loaded]
    assert page_ids, "No page IDs supplied."
    assert len(page_ids) == len(set(page_ids)), "Page IDs must be unique."

    invalid_page_ids = [page_id for page_id in page_ids if page_id_re.fullmatch(page_id) is None]
    assert not invalid_page_ids, invalid_page_ids[:10]
    return page_ids


page_ids = parse_page_ids(PAGE_IDS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pages selected: {len(page_ids):,}")
print(f"Image size: {image_size}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
page_ids[:10]


## Resolve METS Image URLs

Each page ID maps to a local METS file under `data/metadata/<year>/ECON-yyyy-mmdd.mets.xml`. The selected METS file group supplies the image URL for the requested page.

In [ ]:
def mets_path_for_page_id(page_id: str) -> Path:
    match = page_id_re.fullmatch(page_id)
    assert match is not None, page_id
    issue_id = match.group("issue_id")
    year = issue_id[:4]
    return METADATA_ROOT / year / f"ECON-{issue_id}.mets.xml"


def image_href_from_mets(page_id: str, size: str) -> str:
    match = page_id_re.fullmatch(page_id)
    assert match is not None, page_id
    issue_id = match.group("issue_id")
    page_number = match.group("page_number")
    mets_path = mets_path_for_page_id(page_id)
    assert mets_path.exists(), f"Missing METS file for {page_id}: {mets_path}"

    root = ET.parse(mets_path).getroot()
    available_sizes = []
    target_fragment = f"ECON-{issue_id}-{page_number}"

    for file_group in root.findall(".//{*}fileGrp"):
        group_size = str(file_group.attrib.get("USE", "")).upper()
        if group_size:
            available_sizes.append(group_size)
        if group_size != size:
            continue

        for file_node in file_group.findall("{*}file"):
            for location in file_node.findall("{*}FLocat"):
                href = location.attrib.get(XLINK_HREF)
                if href and target_fragment in href:
                    return href

    raise ValueError(
        f"Could not find {size} image URL for {page_id}. "
        f"Available METS fileGrp USE values: {sorted(set(available_sizes))}"
    )


resolved_pages = [
    {
        "page_id": page_id,
        "mets_path": str(mets_path_for_page_id(page_id)),
        "image_size": image_size,
        "image_url": image_href_from_mets(page_id, image_size),
        "output_path": str(OUTPUT_DIR / f"{page_id}.jpg"),
    }
    for page_id in page_ids
]

assert len(resolved_pages) == len(page_ids)
pd.DataFrame(resolved_pages).head(20)


## Authentication

The cookie file must contain `HANID` and `HHAUTHID`, matching the existing METS download workflow. The notebook stops if the server redirects to the Nationallizenzen login flow.

In [ ]:
def load_auth_cookies(path: Path) -> dict[str, str]:
    required = {"HANID", "HHAUTHID"}
    assert path.exists(), f"Missing cookie file: {path.resolve()}"
    cookies = json.loads(path.read_text(encoding="utf-8"))
    missing = sorted(required - set(cookies)) if isinstance(cookies, dict) else sorted(required)
    assert not missing, f"Missing cookie fields in {path}: {missing}"
    assert all(isinstance(cookies[name], str) and cookies[name].strip() for name in required)
    return {name: cookies[name].strip() for name in sorted(required)}


session = requests.Session()
session.headers.update({"User-Agent": "master-thesis-page-image-validation/1.0"})
session.cookies.update(load_auth_cookies(AUTH_COOKIE_PATH))
print(f"Loaded cookies from {AUTH_COOKIE_PATH}")


## Download Images

Images are written as `<page_id>.jpg` in the selected output directory. Existing files are skipped unless `OVERWRITE` is set to `True`.

In [ ]:
def check_response(response: requests.Response, label: str) -> None:
    content_type = response.headers.get("Content-Type", "")
    text_sample = response.text[:5000] if "text" in content_type.lower() else ""
    auth_text = response.url + "\n" + text_sample
    if any(marker in auth_text for marker in AUTH_FAILURE_MARKERS):
        raise RuntimeError(
            f"Authentication failed while fetching {label}. Final URL: {response.url}. "
            f"Refresh {AUTH_COOKIE_PATH} from the logged-in Nationallizenzen session."
        )
    response.raise_for_status()


download_records = []

for index, page in enumerate(resolved_pages, start=1):
    page_id = page["page_id"]
    output_path = Path(page["output_path"])
    status = "downloaded"
    byte_count = None

    if output_path.exists() and not OVERWRITE:
        status = "skipped_existing"
        byte_count = output_path.stat().st_size
    else:
        response = session.get(page["image_url"], timeout=REQUEST_TIMEOUT, allow_redirects=True)
        check_response(response, page_id)
        content_type = response.headers.get("Content-Type", "")
        assert "image" in content_type.lower(), {
            "page_id": page_id,
            "content_type": content_type,
            "url": response.url,
        }
        output_path.write_bytes(response.content)
        byte_count = len(response.content)
        time.sleep(REQUEST_SLEEP_SECONDS)

    download_records.append(
        {
            **page,
            "status": status,
            "bytes": byte_count,
        }
    )
    print(f"[{index}/{len(resolved_pages)}] {page_id}: {status}")

assert len(download_records) == len(page_ids)
assert all(Path(record["output_path"]).exists() for record in download_records)


## Verification

The manifest records the local path and source URL for each downloaded or skipped page. This makes the manual-validation image set reproducible.

In [ ]:
download_manifest_json.write_text(json.dumps(download_records, indent=2) + "\n", encoding="utf-8")

summary = pd.Series(
    {
        "pages_requested": len(page_ids),
        "downloaded": sum(record["status"] == "downloaded" for record in download_records),
        "skipped_existing": sum(record["status"] == "skipped_existing" for record in download_records),
        "manifest": str(download_manifest_json),
        "output_dir": str(OUTPUT_DIR),
    }
)
display(summary)
pd.DataFrame(download_records)
